# Analyzing dbNSFP5.0 alleles with ClinVar

In [1]:
import os
import polars as pl
import pandas as pd
import numpy as np
# import concurrent.futures

aa_single_to_three = {
    'A': 'Ala',  # Alanine
    'R': 'Arg',  # Arginine
    'N': 'Asn',  # Asparagine
    'D': 'Asp',  # Aspartic acid
    'C': 'Cys',  # Cysteine
    'Q': 'Gln',  # Glutamine
    'E': 'Glu',  # Glutamic acid
    'G': 'Gly',  # Glycine
    'H': 'His',  # Histidine
    'I': 'Ile',  # Isoleucine
    'L': 'Leu',  # Leucine
    'K': 'Lys',  # Lysine
    'M': 'Met',  # Methionine
    'F': 'Phe',  # Phenylalanine
    'P': 'Pro',  # Proline
    'S': 'Ser',  # Serine
    'T': 'Thr',  # Threonine
    'W': 'Trp',  # Tryptophan
    'Y': 'Tyr',  # Tyrosine
    'V': 'Val'   # Valine
}

## Get the prediction scores and clean ClinVar annotation

In [ ]:
# chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_data/dbNSFP5.0a/') if "chr" in chr_file and chr_file.endswith(".gz")]
# clin_var_dfs = []
# for chr_num in chr_num_list:
#     df_chr_par = pl.scan_parquet(f"../1_inputs/raw_data/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet")
#     df_chr_par_clinvar = df_chr_par.filter(pl.col("clinvar_id")!='.')
#     df_chr_par_clinvar = df_chr_par_clinvar.collect()
#     clin_var_dfs.append(df_chr_par_clinvar)
# df_clinvar = pl.concat(clin_var_dfs, how="vertical_relaxed")

In [3]:
# ## Create the mapping expression with precise logic for a cleaner ClinVar Annotation
# clinvar_mapping = (
#     # First check for Conflicting interpretations (highest priority)
#     pl.when(
#         pl.col("clinvar_clnsig").str.to_lowercase().str.contains("conflicting")
#     )
#     .then(pl.lit("3_Conflicting"))
    
#     # Then check for Uncertain significance (VUS)
#     .when(pl.col("clinvar_clnsig").str.to_lowercase().str.contains("uncertain"))
#     .then(pl.lit("4_VUS"))
    
#     # # Special case for strings containing both "Uncertain significance" and "Pathogenic"
#     # .when(
#     #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("uncertain") &
#     #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("pathogenic")
#     # )
#     # .then(pl.lit("4_VUS"))  # The dictionary says to classify this as VUS
    
#     # Check for Pathogenic
#     .when(pl.col("clinvar_clnsig").str.to_lowercase().str.contains("pathogenic"))
#     .then(pl.lit("1_Pathogenic"))
    
#     # Check for Benign
#     .when(pl.col("clinvar_clnsig").str.to_lowercase().str.contains("benign"))
#     .then(pl.lit("2_Benign"))
    
#     # Everything else goes to Others
#     .otherwise(pl.lit("5_Others"))
# )

# # Apply the mapping to create a new column
# df_clinvar = df_clinvar.with_columns(clinvar_clnsig_clean=clinvar_mapping)

# # Create a dataframe with unique mappings to check accuracy
# unique_mappings = df_clinvar.select(
#     pl.col("clinvar_clnsig"),
#     pl.col("clinvar_clnsig_clean")
# ).unique()

# # Sort by original value for easier verification
# unique_mappings = unique_mappings.sort("clinvar_clnsig")

# # Print each mapping to verify accuracy
# print("Original Value -> Mapped Value")
# print("-" * 50)
# for row in unique_mappings.rows():
#     print(f"{row[0]} -> {row[1]}")

In [ ]:
# parquet_path = f"../dbnsfp_data/dbNSFP5.0/dbNSFP5.0a/dbNSFP5.0a_variant.clin_var_annot.parquet"
# df_clinvar.write_parquet(parquet_path)

## Preprocess the ClinVar Annotation

In [7]:
df_clinvar = pl.scan_parquet(f"../1_inputs/raw_data/dbNSFP5.0a/dbNSFP5.0a_variant.clin_var_annot.parquet")
# df_clinvar.filter(pl.col("aaalt").str.len_chars() == 1).collect()

In [ ]:
# For testing purposes, limit to a small subset:
df_clin_subset = df_clinvar.limit(10)

# Collect the subset into an eager DataFrame for inspection
df_clin_subset = df_clin_subset.collect()

# Automatically detect columns with semicolon-separated patterns.
# We only check columns of Utf8 (string) type.
pattern_cols = [
    col for col in df_clinvar.collect_schema() 
    if df_clin_subset[col].dtype == pl.Utf8 and df_clin_subset[col].str.contains(";").any()
]
print("Columns containing ';':", pattern_cols)

# Now, suppose you have a function that takes a DataFrame and splits/explodes these columns.
# You can test it on the subset first:
def split_and_explode(df: pl.DataFrame, columns: list) -> pl.DataFrame:
    # Convert each target column from a string to a list (using ";" as separator)
    df = df.with_columns([pl.col(col).str.split(";").alias(col) for col in columns])
    # Explode all the target columns simultaneously.
    return df.explode(columns)

df_clinvar_exploded = df_clinvar.with_columns(
    [pl.col(col).str.split(";").alias(col) for col in pattern_cols]
).explode(pattern_cols)

Columns containing ';': ['aapos', 'genename', 'Ensembl_geneid', 'Ensembl_transcriptid', 'Ensembl_proteinid', 'Uniprot_acc', 'Uniprot_entry', 'HGVSc_snpEff', 'HGVSp_snpEff', 'HGVSc_VEP', 'HGVSp_VEP', 'APPRIS', 'GENCODE_basic', 'TSL', 'VEP_canonical', 'MANE', 'cds_strand', 'refcodon', 'codonpos', 'codon_degeneracy', 'SIFT_score', 'SIFT_pred', 'SIFT4G_score', 'SIFT4G_pred', 'Polyphen2_HDIV_score', 'Polyphen2_HDIV_pred', 'Polyphen2_HVAR_score', 'Polyphen2_HVAR_pred', 'MutationTaster_score', 'MutationTaster_pred', 'MutationTaster_model', 'MutationTaster_trees_benign', 'MutationTaster_trees_deleterious', 'MutationAssessor_score', 'MutationAssessor_pred', 'PROVEAN_score', 'PROVEAN_pred', 'VEST4_score', 'MetaRNN_score', 'MetaRNN_pred', 'REVEL_score', 'MutPred_score', 'MutPred_protID', 'MutPred_AAchange', 'MutPred_Top5features', 'MVP_score', 'gMVP_score', 'MPC_score', 'DEOGEN2_score', 'DEOGEN2_pred', 'LIST-S2_score', 'LIST-S2_pred', 'VARITY_R_score', 'VARITY_ER_score', 'VARITY_R_LOO_score', 'VA

In [ ]:
# Once you’re happy with the behavior on the subset,
# you can apply it to the entire lazy DataFrame.
# Assume ldf is your lazy DataFrame.
# Randomly sample 1000 rows from the exploded DataFrame
# df_clinvar_sampled = df_clinvar_exploded.collect().sample(n=1000)

df_clinvar_sampled = df_clinvar_exploded.collect().sample(n=100, seed=42)

variant_annot = [
    "hg19_chr","genename","Ensembl_geneid","Ensembl_proteinid",
    "Uniprot_acc","Uniprot_entry",
    "aapos","aaref","aaref_3","aaalt","aaalt_3",#"aa_change",
    "Ensembl_transcriptid","hg19_pos(1-based)","HGVSc_snpEff","HGVSp_snpEff",
    "HGVSc_VEP","HGVSp_VEP",
]

predictor_algos = [
    "PROVEAN_score","PROVEAN_converted_rankscore","PROVEAN_pred",
    "ESM1b_score","ESM1b_rankscore","ESM1b_pred",
    "AlphaMissense_score","AlphaMissense_rankscore","AlphaMissense_pred"
] + [col for col in df_clinvar_sampled.collect_schema().names() if "clinvar" in col]

predictor_algos

['PROVEAN_score',
 'PROVEAN_converted_rankscore',
 'PROVEAN_pred',
 'ESM1b_score',
 'ESM1b_rankscore',
 'ESM1b_pred',
 'AlphaMissense_score',
 'AlphaMissense_rankscore',
 'AlphaMissense_pred',
 'clinvar_id',
 'clinvar_clnsig',
 'clinvar_trait',
 'clinvar_review',
 'clinvar_hgvs',
 'clinvar_var_source',
 'clinvar_MedGen_id',
 'clinvar_OMIM_id',
 'clinvar_Orphanet_id',
 'clinvar_clnsig_clean']

In [113]:
df_clinvar_sampled = df_clinvar_sampled.select(pl.col(variant_annot+predictor_algos))
print(df_clinvar_sampled)

shape: (100, 36)
┌──────────┬──────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ hg19_chr ┆ genename ┆ Ensembl_ge ┆ Ensembl_p ┆ … ┆ clinvar_M ┆ clinvar_O ┆ clinvar_O ┆ clinvar_c │
│ ---      ┆ ---      ┆ neid       ┆ roteinid  ┆   ┆ edGen_id  ┆ MIM_id    ┆ rphanet_i ┆ lnsig_cle │
│ str      ┆ str      ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ d         ┆ an        │
│          ┆          ┆ str        ┆ str       ┆   ┆ str       ┆ str       ┆ ---       ┆ ---       │
│          ┆          ┆            ┆           ┆   ┆           ┆           ┆ str       ┆ str       │
╞══════════╪══════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 11       ┆ SERPING1 ┆ ENSG000001 ┆ ENSP00000 ┆ … ┆ C3661900  ┆ .         ┆ .         ┆ 4_VUS     │
│          ┆          ┆ 49131      ┆ 478572    ┆   ┆           ┆           ┆           ┆           │
│ 2        ┆ MSH6     ┆ ENSG000001 ┆ ENSP00000 ┆ … ┆ C0009405  ┆ .        

In [63]:
## Merge the AI scores
import sys
import os
sys.path.insert(1, "scripts/")

import extract_plddt_scores as get_plddt
import calculate_rsa_scores as get_rsa

In [114]:
ldf = df_clinvar_sampled
ldf

hg19_chr,genename,Ensembl_geneid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,aapos,aaref,aaref_3,aaalt,aaalt_3,Ensembl_transcriptid,hg19_pos(1-based),HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,PROVEAN_score,PROVEAN_converted_rankscore,PROVEAN_pred,ESM1b_score,ESM1b_rankscore,ESM1b_pred,AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,clinvar_id,clinvar_clnsig,clinvar_trait,clinvar_review,clinvar_hgvs,clinvar_var_source,clinvar_MedGen_id,clinvar_OMIM_id,clinvar_Orphanet_id,clinvar_clnsig_clean
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""11""","""SERPING1""","""ENSG00000149131""","""ENSP00000478572""","""A0A087WUD9""","""A0A087WUD9_HUMAN""","""389""","""A""","""Ala""","""S""","""Ser""","""ENST00000619430""","""57381920""","""c.*98G>T""",""".""","""c.1165G>T""","""p.Ala389Ser""",""".""","""0.25939""",""".""",""".""","""0.24363""",""".""",""".""","""0.37834""",""".""","""1511723""","""Uncertain_significance""","""not_provided""","""criteria_provided,_single_subm…","""NC_000011.10:g.57614447G>T""",""".""","""C3661900""",""".""",""".""","""4_VUS"""
"""2""","""MSH6""","""ENSG00000116062""","""ENSP00000514752""","""A0A8V8TQW9""","""A0A8V8TQW9_HUMAN""","""978""","""Q""","""Gln""","""H""","""His""","""ENST00000700004""","""48028056""",""".""",""".""","""c.2934G>C""","""p.Gln978His""",""".""","""0.80975""",""".""",""".""","""0.73401""",""".""",""".""","""0.92384""",""".""","""1424001""","""Uncertain_significance""","""Hereditary_nonpolyposis_colore…","""criteria_provided,_single_subm…","""NC_000002.12:g.47800917G>C""",""".""","""C0009405""",""".""",""".""","""4_VUS"""
"""19""","""RPS19""","""ENSG00000105372""","""ENSP00000469228""","""P39019""","""RS19_HUMAN""","""-1""",""".""",""".""",""".""",""".""","""ENST00000600467""","""42375417""","""c.412-2A>G""",""".""","""c.412-2A>G""",""".""",""".""",""".""",""".""",""".""",""".""",""".""",""".""",""".""",""".""","""2780440""","""Pathogenic""","""Diamond-Blackfan_anemia""","""criteria_provided,_single_subm…","""NC_000019.10:g.41871349A>G""",""".""","""C1260899""","""PS105650""","""124""","""1_Pathogenic"""
"""22""","""LARGE1""","""ENSG00000133424""","""ENSP00000502006""","""A0A6Q8PFY8""","""A0A6Q8PFY8_HUMAN""","""200""","""N""","""Asn""","""S""","""Ser""","""ENST00000476315""","""34000437""",""".""",""".""","""c.599A>G""","""p.Asn200Ser""",""".""","""0.30931""",""".""",""".""","""0.53463""",""".""",""".""","""0.12522""",""".""","""1429168""","""Uncertain_significance""","""not_provided|Muscular_dystroph…","""criteria_provided,_multiple_su…","""NC_000022.11:g.33604451T>C""",""".""","""C3661900|C1837229""","""608840""","""98894""","""4_VUS"""
"""16""","""CACNA1H""","""ENSG00000196557""","""ENSP00000518778""","""A0AAA9YHG8""","""A0AAA9YHG8_HUMAN""","""854""","""R""","""Arg""","""W""","""Trp""","""ENST00000711493""","""1255222""",""".""",""".""","""c.2560C>T""","""p.Arg854Trp""",""".""","""0.86639""",""".""",""".""","""0.75624""",""".""",""".""","""0.35393""",""".""","""1437005""","""Likely_benign""","""Idiopathic_generalized_epileps…","""criteria_provided,_single_subm…","""NC_000016.10:g.1205222C>T""",""".""","""C0270850|C4310756""","""600669|PS600669|617027""","""642671""","""2_Benign"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2""","""BARD1""","""ENSG00000138376""","""ENSP00000480470""","""Q99728-2""","""BARD1_HUMAN""","""24""","""P""","""Pro""","""A""","""Ala""","""ENST00000617164""","""215674224""","""c.70C>G""","""p.Pro24Ala""","""c.70C>G""","""p.Pro24Ala""",""".""","""0.33573""",""".""","""-2.21""","""0.05834""","""T""","""0.063""","""0.05165""","""LB""","""1757163""","""Uncertain_significance""","""BARD1-related_disorder|Heredit…","""criteria_provided,_multiple_su…","""NC_000002.12:g.214809500G>C""",""".""","""C0027672""",""".""","""140162""","""4_VUS"""
"""17""","""BRCA1""","""ENSG00000012048""","""ENSP00000419103""","""P38398""","""BRCA

In [ ]:
parquet_dir_plddt = "../3_outputs/processed_data/plddt_scores"
parquet_files_plddt = [f"{parquet_dir_plddt}/{par_f}" for par_f in os.listdir(parquet_dir_plddt)]

def get_plddt_score(pdb_id, aapos, aaref, parquet_files_plddt=parquet_files_plddt):
    # 1. Get all scores for a specific protein
    df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F1-model_v4"])
    if df is None or df.empty:
        return np.nan
    
    try:
        if int(aapos) in df.index and df.loc[int(aapos), "residue_name"] == aaref:
            return df.loc[int(aapos), "plddt"]
    except:
        pass
    return np.nan
    

parquet_dir_rsa = "../3_outputs/processed_data/rsa_scores"
parquet_files_rsa = [f"{parquet_dir_rsa}/{par_f}" for par_f in os.listdir(parquet_dir_rsa)]

def get_rsa_score(pdb_id, aapos, aaref, parquet_files_rsa=parquet_files_rsa):
    # 1. Get all scores for a specific protein
    df = get_rsa.get_structure_rsa(parquet_files_rsa, pdb_id=f"AF-{pdb_id}-F1-model_v4")
    if df is None or df.empty:
        return np.nan
    
    try:
        if int(aapos) in df.index and df.loc[int(aapos), "residue_name"] == aaref:
            return df.loc[int(aapos), "rsa"]
    except:
        pass
    return np.nan
    
## testing uniprot score mapping
df_transformed = (
    ldf.with_columns(
        pl.struct(pl.col(["Uniprot_acc", "aapos", "aaref_3"]))
          .map_elements(
              lambda x: get_plddt_score(
                  pdb_id=x["Uniprot_acc"].split('-')[0],
                  aapos=int(x["aapos"]),
                  aaref=x["aaref_3"].upper(),
                  parquet_files_plddt=parquet_files_plddt
              ),
              return_dtype=pl.Float64
          )
          .alias("plddt"),
        pl.struct(pl.col(["Uniprot_acc", "aapos", "aaref_3"]))
          .map_elements(
              lambda x: get_rsa_score(
                  pdb_id=x["Uniprot_acc"].split('-')[0],
                  aapos=int(x["aapos"]),
                  aaref=x["aaref_3"].upper(),
                  parquet_files_rsa=parquet_files_rsa
              ),
              return_dtype=pl.Float64
          )
          .alias("rsa"),
        # pl.struct(pl.col(["Ensembl_proteinid", "aapos", "aaref_3", "aaalt"]))
        #   .map_elements(
        #       lambda x: get_rsa_score(
        #           prot_id=x["Ensembl_proteinid"],
        #           aapos=int(x["aapos"]),
        #           aaref=x["aaref_3"],
        #           aa=x["aaalt"]
        #       ),
        #       return_dtype=pl.Float64
        #   )
        #   .alias("provean_alt")
    )
)

df_transformed

hg19_chr,genename,Ensembl_geneid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,aapos,aaref,aaref_3,aaalt,aaalt_3,Ensembl_transcriptid,hg19_pos(1-based),HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,PROVEAN_score,PROVEAN_converted_rankscore,PROVEAN_pred,ESM1b_score,ESM1b_rankscore,ESM1b_pred,AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,clinvar_id,clinvar_clnsig,clinvar_trait,clinvar_review,clinvar_hgvs,clinvar_var_source,clinvar_MedGen_id,clinvar_OMIM_id,clinvar_Orphanet_id,clinvar_clnsig_clean,plddt,rsa
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64
"""11""","""SERPING1""","""ENSG00000149131""","""ENSP00000478572""","""A0A087WUD9""","""A0A087WUD9_HUMAN""","""389""","""A""","""Ala""","""S""","""Ser""","""ENST00000619430""","""57381920""","""c.*98G>T""",""".""","""c.1165G>T""","""p.Ala389Ser""",""".""","""0.25939""",""".""",""".""","""0.24363""",""".""",""".""","""0.37834""",""".""","""1511723""","""Uncertain_significance""","""not_provided""","""criteria_provided,_single_subm…","""NC_000011.10:g.57614447G>T""",""".""","""C3661900""",""".""",""".""","""4_VUS""",NaN,NaN
"""2""","""MSH6""","""ENSG00000116062""","""ENSP00000514752""","""A0A8V8TQW9""","""A0A8V8TQW9_HUMAN""","""978""","""Q""","""Gln""","""H""","""His""","""ENST00000700004""","""48028056""",""".""",""".""","""c.2934G>C""","""p.Gln978His""",""".""","""0.80975""",""".""",""".""","""0.73401""",""".""",""".""","""0.92384""",""".""","""1424001""","""Uncertain_significance""","""Hereditary_nonpolyposis_colore…","""criteria_provided,_single_subm…","""NC_000002.12:g.47800917G>C""",""".""","""C0009405""",""".""",""".""","""4_VUS""",NaN,NaN
"""19""","""RPS19""","""ENSG00000105372""","""ENSP00000469228""","""P39019""","""RS19_HUMAN""","""-1""",""".""",""".""",""".""",""".""","""ENST00000600467""","""42375417""","""c.412-2A>G""",""".""","""c.412-2A>G""",""".""",""".""",""".""",""".""",""".""",""".""",""".""",""".""",""".""",""".""","""2780440""","""Pathogenic""","""Diamond-Blackfan_anemia""","""criteria_provided,_single_subm…","""NC_000019.10:g.41871349A>G""",""".""","""C1260899""","""PS105650""","""124""","""1_Pathogenic""",NaN,NaN
"""22""","""LARGE1""","""ENSG00000133424""","""ENSP00000502006""","""A0A6Q8PFY8""","""A0A6Q8PFY8_HUMAN""","""200""","""N""","""Asn""","""S""","""Ser""","""ENST00000476315""","""34000437""",""".""",""".""","""c.599A>G""","""p.Asn200Ser""",""".""","""0.30931""",""".""",""".""","""0.53463""",""".""",""".""","""0.12522""",""".""","""1429168""","""Uncertain_significance""","""not_provided|Muscular_dystroph…","""criteria_provided,_multiple_su…","""NC_000022.11:g.33604451T>C""",""".""","""C3661900|C1837229""","""608840""","""98894""","""4_VUS""",NaN,NaN
"""16""","""CACNA1H""","""ENSG00000196557""","""ENSP00000518778""","""A0AAA9YHG8""","""A0AAA9YHG8_HUMAN""","""854""","""R""","""Arg""","""W""","""Trp""","""ENST00000711493""","""1255222""",""".""",""".""","""c.2560C>T""","""p.Arg854Trp""",""".""","""0.86639""",""".""",""".""","""0.75624""",""".""",""".""","""0.35393""",""".""","""1437005""","""Likely_benign""","""Idiopathic_generalized_epileps…","""criteria_provided,_single_subm…","""NC_000016.10:g.1205222C>T""",""".""","""C0270850|C4310756""","""600669|PS600669|617027""","""642671""","""2_Benign""",NaN,NaN
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2""","""BARD1""","""ENSG00000138376""","""ENSP00000480470""","""Q99728-2""","""BARD1_HUMAN""","""24""","""P""","""Pro""","""A""","""Ala""","""ENST00000617164""","""215674224""","""c.70C>G""","""p.Pro24Ala""","""c.70C>G""","""p.Pro24Ala""",""".""","""0.33573""",""".""","""-2.21""","""0.05834""","""T""","""0.063""","""0.05165""","""LB""","""1757163""","""Uncertain_significance""","""BARD1-related_disorder|Heredit…","""criteria_provided,_multiple_su…","""NC_000002.12:g.214809500G>C""",""".""","""C0027672""",""".""","""140162""","""4_VUS""",46.25,0.698529
"""17"""

### Long protein sequences problem

In [ ]:
pdb_id = "Q8WZ42"
df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F2-model_v4"])
df
# df[df["residue_num"]==311]
# df[df["residue_num"]<=311].tail(10)

,pdb_id,chain_id,residue_num,residue_name,plddt
0,AF-Q8WZ42-F2-model_v4,A,1,ALA,39.30
1,AF-Q8WZ42-F2-model_v4,A,2,LYS,38.72
2,AF-Q8WZ42-F2-model_v4,A,3,LYS,42.24
3,AF-Q8WZ42-F2-model_v4,A,4,THR,41.45
4,AF-Q8WZ42-F2-model_v4,A,5,LYS,40.27
...,...,...,...,...,...
1395,AF-Q8WZ42-F2-model_v4,A,1396,ILE,82.81
1396,AF-Q8WZ42-F2-model_v4,A,1397,VAL,73.36
1397,AF-Q8WZ42-F2-model_v4,A,1398,PRO,67.68
1398,AF-Q8WZ42-F2-model_v4,A,1399,HIS,59.86


In [ ]:
import polars as pl
import re
from typing import Optional, Tuple

def process_accession_file(input_csv: str, output_parquet: str):
    """
    Process large accession CSV into optimized Parquet format with interval indexing
    [Uses Polars' streaming and chunk processing for memory efficiency]
    """
    # Read with schema optimization and streaming mode
    df = pl.scan_csv(
        input_csv,
        separator=',',
        n_rows=5, has_header=False, 
        new_columns=["uniprot_id","first_res_idx","last_res_idx","af_db_identifier","latest_version"]
    )
    
    # Add subfile index extraction (F1, F2 from AF-ID)
    processed = df.with_columns(
        pl.col("af_db_identifier")
        .str.extract(r"-F(\d+).*", 1)  # Capture all digits after F until end
        .alias("subfile_index")
    ).sort(["uniprot_id", "first_res_idx"])

    print(processed.collect())
    
    # Sort by uniprot_id and first_res_idx for interval search
    processed = processed.sort(["uniprot_id", "first_res_idx"])
    
    # Write optimized Parquet with partitioning
    processed.sink_parquet(
        output_parquet,
        row_group_size=10000,
        compression="zstd"
    )


def find_af_file(parquet_path: str, uniprot_id: str, residue_pos: int) -> Optional[Tuple[str, int]]:
    """
    Find corresponding AF file and subfile index for a residue position
    [Uses binary search on sorted intervals for O(log n) performance]
    """
    # Load only relevant partitions
    df = pl.scan_parquet(parquet_path)
    
    result = df.filter(
        (pl.col("uniprot_id") == uniprot_id) &
        (pl.col("first_res_idx") <= residue_pos) &
        (pl.col("last_res_idx") >= residue_pos)
    ).collect()
    
    if result.height == 1:
        row = result.row(0, named=True)
        return (row["af_db_identifier"], row["subfile_index"])
    return (row["af_db_identifier"], None)


def find_af_entry(parquet_path: str, uniprot_id: str, residue_pos: int) -> Optional[dict]:
    """
    Find complete AF file entry with residue range checking
    Returns full row as dictionary if found
    """
    df = pl.scan_parquet(parquet_path)
    result = df.filter(
        (pl.col("uniprot_id") == uniprot_id) &
        (pl.col("first_res_idx") <= residue_pos) &
        (pl.col("last_res_idx") >= residue_pos)
    ).collect()
    
    return result.row(0, named=True) if result.height == 1 else None


def calculate_local_position(af_entry: dict, residue_pos: int) -> int:
    """
    Calculate position within the subfile using 1-based indexing
    Formula: local_pos = residue_pos - first_res_idx + 1
    """
    return residue_pos - af_entry["first_res_idx"] + 1


# Integrated usage example
def get_af_file_and_position(parquet_path: str, uniprot_id: str, residue_pos: int):
    entry = find_af_entry(parquet_path, uniprot_id, residue_pos)
    if not entry:
        return None
        
    return {
        "af_id": entry["af_db_identifier"],
        "subfile_index": entry["subfile_index"],
        "local_position": calculate_local_position(entry, residue_pos),
        "global_range": (entry["first_res_idx"], entry["last_res_idx"])
    }


def get_plddt_score(pdb_id: str, aa_pos: int, aaref: str, 
                    parquet_files_plddt: str, accession_parquet: str, 
                    plddt_module) -> float:
    """Enhanced pLDDT lookup with dynamic AF model selection and position conversion"""
    # 1. Find correct AlphaFold file and local position
    af_entry = find_af_entry(accession_parquet, pdb_id, aa_pos)
    
    if not af_entry:
        return np.nan
    
    # 2. Convert global position to subfile-local index
    local_pos = aa_pos - af_entry["first_res_idx"] + 1  # [1](@ref)
    
    # 3. Query pLDDT data from correct AF model
    df = plddt_module.query_plddt(
        parquet_files_plddt, 
        pdb_ids=[af_entry["af_db_identifier"]]
    )
    
    # 4. Validate and return score
    try:
        if (local_pos in df.index and 
            df.loc[local_pos, "residue_name"] == aaref):
            return df.loc[local_pos, "plddt"]
    except (KeyError, TypeError):
        pass
    return np.nan


# Usage example
process_accession_file("../1_inputs/raw_data/alphafold/accession_ids.csv", 
                       "../3_outputs/processed_data/af_id_mapping/af_mapping.parquet")
print(find_af_file("../3_outputs/processed_data/af_id_mapping/af_mapping.parquet", 
                   "A0A2I1YHU5", 500))
# Output: ("AF-A0A2I1YHU5-F1", 1)

shape: (5, 6)
┌────────────┬───────────────┬──────────────┬──────────────────┬────────────────┬───────────────┐
│ uniprot_id ┆ first_res_idx ┆ last_res_idx ┆ af_db_identifier ┆ latest_version ┆ subfile_index │
│ ---        ┆ ---           ┆ ---          ┆ ---              ┆ ---            ┆ ---           │
│ str        ┆ i64           ┆ i64          ┆ str              ┆ i64            ┆ str           │
╞════════════╪═══════════════╪══════════════╪══════════════════╪════════════════╪═══════════════╡
│ A0A1G9ZVL8 ┆ 1             ┆ 268          ┆ AF-A0A1G9ZVL8-F1 ┆ 4              ┆ 1             │
│ A0A2I1YHU5 ┆ 1             ┆ 933          ┆ AF-A0A2I1YHU5-F1 ┆ 4              ┆ 1             │
│ A0A5H2Z360 ┆ 1             ┆ 342          ┆ AF-A0A5H2Z360-F1 ┆ 4              ┆ 1             │
│ A0A6L5B7P9 ┆ 1             ┆ 275          ┆ AF-A0A6L5B7P9-F1 ┆ 4              ┆ 1             │
│ A0A7G6RQS6 ┆ 1             ┆ 316          ┆ AF-A0A7G6RQS6-F1 ┆ 4              ┆ 1             │
└─────

/tmp/ipykernel_2960000/848040726.py:31: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  processed.sink_parquet(


In [18]:
af_accession_id = pl.scan_csv("../1_inputs/raw_data/alphafold/accession_ids.csv",
                              n_rows=5, has_header=False, 
                              new_columns=["uniprot_id","first_res_idx","last_res_idx","af_db_identifier","latest_version"])
af_accession_id.collect()

uniprot_id,first_res_idx,last_res_idx,af_db_identifier,latest_version
str,i64,i64,str,i64
"""A0A2I1YHU5""",1,933,"""AF-A0A2I1YHU5-F1""",4
"""A0A5H2Z360""",1,342,"""AF-A0A5H2Z360-F1""",4
"""A0A6L5B7P9""",1,275,"""AF-A0A6L5B7P9-F1""",4
"""A0A1G9ZVL8""",1,268,"""AF-A0A1G9ZVL8-F1""",4
"""A0A7G6RQS6""",1,316,"""AF-A0A7G6RQS6-F1""",4


In [12]:
# Configuration
csv_path = "../1_inputs/raw_data/alphafold/accession_ids.csv"
pickle_path = "../3_outputs/af_id_mapping/af_mapping.pickle"

# Initialize the mapper (will create pickle if it doesn't exist)
print("Initializing AlphaFold file mapper...")
mapper = AFFileMapper(csv_path=csv_path, pickle_path=pickle_path)

Initializing AlphaFold file mapper...
Processing CSV file: ../1_inputs/raw_data/alphafold/accession_ids.csv


KeyboardInterrupt: 

In [ ]:
af_base_dir = "/path/to/alphafold/pdbs"
# Example: Look up a specific residue
uniprot_id = "A0A2I1YHU5"
residue_pos = 500

print(f"\nLooking up {uniprot_id} residue {residue_pos}...")
residue = get_alphafold_residue(mapper, uniprot_id, residue_pos, af_base_dir)

if residue:
    print(f"Found residue: {residue}")
else:
    print(f"Residue not found")

# Example: Get all AlphaFold files for a UniProt ID
print(f"\nGetting all AlphaFold files for {uniprot_id}...")
af_files = mapper.get_all_af_files_for_uniprot(uniprot_id)

if af_files:
    print(f"Found {len(af_files)} AlphaFold files:")
    for af_file in af_files:
        print(f"  {af_file['af_file']} (Residues {af_file['first_res_idx']}-{af_file['last_res_idx']})")
else:
    print(f"No AlphaFold files found for {uniprot_id}")